# FloorGen: Retrieval-Augmented Generative Floorplan Synthesis
### 100% Free Google Colab / Kaggle T4 GPU Training Notebook (Zero Monetary Cost)

This notebook trains and evaluates the FloorGen RAG-Conditioned Floorplan Diffusion model on Google Colab (Free T4 GPU, 16GB VRAM) or Kaggle. No payment or paid subscription is required.

In [ ]:
# 1. Verify Free GPU environment
!nvidia-smi
!pip install -q torch torchvision networkx shapely matplotlib faiss-cpu sentence-transformers tqdm scipy

In [ ]:
# 2. Clone FloorGen repository (or use current workspace)
!git clone https://github.com/mrinal22258/floorgen.git
%cd floorgen

In [ ]:
# 3. Ingest and build canonical dataset
from floorgen.data.scripts.generate_sample_data import generate_dataset
from floorgen.data.scripts.parse_rplan import load_plans_from_dir

plans = generate_dataset(num_samples=500, output_dir="data/processed")
print(f"Loaded {len(plans)} canonical floorplans for training.")

In [ ]:
# 4. Build RAG Index (FAISS + Graph Store)
from floorgen.rag.retriever import FloorplanRAGRetriever

retriever = FloorplanRAGRetriever()
retriever.index_corpus(plans)
print("RAG vector and graph index built successfully.")

In [ ]:
# 5. Train RAG-Conditioned Diffusion Core on GPU
from floorgen.models.train import train_rag_diffusion

model = train_rag_diffusion(
    data_dir="data/processed",
    epochs=25,
    batch_size=32,
    lr=1e-3,
    save_path="checkpoints/rag_diffusion_colab.pt"
)

In [ ]:
# 6. Run Benchmark & Evaluation (FID, KID, Graph Edit Distance, Realism)
from floorgen.eval.benchmark import run_benchmark

benchmark_results = run_benchmark(data_dir="data/processed", num_test_samples=50)

In [ ]:
# 7. Interactive Sample Synthesis and Vector SVG Export
from floorgen.demo.cli import main as run_cli
import sys

sys.argv = [
    "cli.py",
    "--rooms", "living_room,master_bedroom,second_bedroom,bathroom,kitchen,balcony",
    "--brief", "Sun-drenched south-facing 2-bedroom with balcony and private ensuite",
    "--top_k", "5",
    "--output", "colab_generated_plan.svg"
]
run_cli()

# Display SVG in notebook
from IPython.display import SVG, display
display(SVG(filename="colab_generated_plan.svg"))